In [0]:
# Read from processed (Silver), not from raw or in-memory dataframes
df_employees_clean = spark.read.format("delta").load("abfss://silver@hr1storage1.dfs.core.windows.net/employees/")
df_attendees_clean = spark.read.format("delta").load("abfss://silver@hr1storage1.dfs.core.windows.net/attendees/")
df_performance = spark.read.format("delta").load("abfss://silver@hr1storage1.dfs.core.windows.net/performance/")

In [0]:
print("Employees:", df_employees_clean.count())
print("Attendees:", df_attendees_clean.count())
print("Performance:", df_performance.count())

Employees: 20
Attendees: 120
Performance: 20


In [0]:
df_dim_employee = df_employees_clean.select(
    "EmployeeID", "Name", "Department", "Position", "HireDate_parsed", "Salary"
)

df_dim_employee.show(25, truncate=False)

+----------+------------------+----------+--------------------+---------------+---------+
|EmployeeID|Name              |Department|Position            |HireDate_parsed|Salary   |
+----------+------------------+----------+--------------------+---------------+---------+
|1001      |Jason Summers     |hr        |HR Specialist       |2026-01-16     |18686    |
|1002      |Rachel Herman     |Sales     |Account Manager     |2023-08-23     |18698.46 |
|1003      |Tony Wolfe        |Finance   |Financial Analyst   |2021-10-07     |23191    |
|1004      |John Lewis        |Sales     |Sales Manager       |2026-04-28     |23714.8  |
|1005      |Colleen Hall MD   |HR        |HR Specialist       |2023-01-02     |$13,591  |
|1006      |Amber Yoder       |Finance   |Accountant          |2022-11-14     |18679.79 |
|1007      |Michael Reid      |MISSING   |Sales Representative|2021-02-01     |13038.92 |
|1008      |Bryan Carey       |IT        |Software Developer  |2024-05-02     |11655.06 |
|1009     

In [0]:
from pyspark.sql.functions import col

df_fact_attendance = df_attendees_clean.select(
    "EmployeeID",
    "Month_standardized",
    "DaysPresent",
    "DaysAbsent",
    "LateArrivals",
    "DaysAbsent_flagged_invalid"
).withColumnRenamed("Month_standardized", "Month")

# Cast measures to proper numeric types (they came in from Delta likely already typed, but confirm)
df_fact_attendance = df_fact_attendance \
    .withColumn("DaysPresent", col("DaysPresent").cast("int")) \
    .withColumn("DaysAbsent", col("DaysAbsent").cast("int")) \
    .withColumn("LateArrivals", col("LateArrivals").cast("int"))

df_fact_attendance.show(25, truncate=False)

+----------+-----+-----------+----------+------------+--------------------------+
|EmployeeID|Month|DaysPresent|DaysAbsent|LateArrivals|DaysAbsent_flagged_invalid|
+----------+-----+-----------+----------+------------+--------------------------+
|1001      |1    |16         |6         |0           |false                     |
|1001      |2    |19         |1         |1           |false                     |
|1001      |3    |18         |1         |5           |false                     |
|1001      |4    |19         |1         |0           |false                     |
|1001      |5    |21         |NULL      |4           |true                      |
|1001      |6    |17         |5         |4           |false                     |
|1002      |1    |16         |5         |NULL        |false                     |
|1002      |2    |22         |0         |0           |false                     |
|1002      |3    |18         |NULL      |NULL        |true                      |
|1002      |4   

In [0]:
from pyspark.sql.functions import year, quarter

df_fact_performance = df_performance.select(
    "EmployeeID",
    "ReviewDate",
    "PerformanceScore_numeric",
    "PerformanceScore_imputed",
    "ManagerComment"
).withColumn("ReviewYear", year("ReviewDate")) \
 .withColumn("ReviewQuarter", quarter("ReviewDate")) \
 .withColumnRenamed("PerformanceScore_numeric", "PerformanceScore")

df_fact_performance.show(25, truncate=False)

+----------+-------------------+----------------+------------------------+-----------------+----------+-------------+
|EmployeeID|ReviewDate         |PerformanceScore|PerformanceScore_imputed|ManagerComment   |ReviewYear|ReviewQuarter|
+----------+-------------------+----------------+------------------------+-----------------+----------+-------------+
|1006.0    |2026-03-31 00:00:00|3.6             |false                   |Satisfactory     |2026      |1            |
|1007.0    |2026-03-31 00:00:00|3.56            |true                    |Good             |2026      |1            |
|1008.0    |2026-03-31 00:00:00|3.56            |true                    |Outstanding      |2026      |1            |
|1009.0    |2026-03-31 00:00:00|4.6             |false                   |Needs Improvement|2026      |1            |
|1010.0    |2026-03-31 00:00:00|3.2             |false                   |Satisfactory     |2026      |1            |
|1001.0    |2026-03-31 00:00:00|3.6             |false  

In [0]:
dim_ids = set(row.EmployeeID for row in df_dim_employee.select("EmployeeID").distinct().collect())

attendance_ids = set(row.EmployeeID for row in df_fact_attendance.select("EmployeeID").distinct().collect())
performance_ids = set(row.EmployeeID for row in df_fact_performance.select("EmployeeID").distinct().collect())

print("Attendance IDs not in dim_employee:", attendance_ids - dim_ids)
print("Performance IDs not in dim_employee:", performance_ids - dim_ids)

Attendance IDs not in dim_employee: set()
Performance IDs not in dim_employee: {1001.0, 1002.0, 1003.0, 1004.0, 1005.0, 1006.0, 1007.0, 1008.0, 1009.0, 1010.0, 1011.0, 1012.0, 1013.0, 1014.0, 1015.0, 1016.0, 1017.0, 1018.0, 1019.0, 1020.0}


In [0]:
df_dim_employee.write.mode("overwrite").format("delta") \
    .save("abfss://gold@hr1storage1.dfs.core.windows.net/dim_employee/")

df_fact_attendance.write.mode("overwrite").format("delta") \
    .save("abfss://gold@hr1storage1.dfs.core.windows.net/fact_attendance/")

df_fact_performance.write.mode("overwrite").format("delta") \
    .save("abfss://gold@hr1storage1.dfs.core.windows.net/fact_performance/")

print("Gold layer written.")

Gold layer written.


In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS hr_analytics_databricks.default.dim_employee
USING DELTA
LOCATION 'abfss://gold@hr1storage1.dfs.core.windows.net/dim_employee/'
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS hr_analytics_databricks.default.fact_attendance
USING DELTA
LOCATION 'abfss://gold@hr1storage1.dfs.core.windows.net/fact_attendance/'
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS hr_analytics_databricks.default.fact_performance
USING DELTA
LOCATION 'abfss://gold@hr1storage1.dfs.core.windows.net/fact_performance/'
""")

DataFrame[]

In [0]:
spark.sql("SHOW TABLES IN hr_analytics_databricks.default").show()

+--------+----------------+-----------+
|database|       tableName|isTemporary|
+--------+----------------+-----------+
| default|    dim_employee|      false|
| default| fact_attendance|      false|
| default|fact_performance|      false|
+--------+----------------+-----------+



In [0]:
%sql
SELECT
    EmployeeID,
    COUNT(*) AS count
FROM hr_analytics_databricks.default.dim_employee
GROUP BY EmployeeID
HAVING COUNT(*) > 1
ORDER BY count DESC;

EmployeeID,count


In [0]:
df_attendance = spark.read.format("delta").load("abfss://gold@hr1storage1.dfs.core.windows.net/fact_attendance/")

from pyspark.sql import functions as F

# Per-employee attendance rate, calculated directly — same formula as your Power BI measure
df_rate = df_attendance.groupBy("EmployeeID").agg(
    F.sum("DaysPresent").alias("TotalPresent"),
    F.sum("DaysAbsent").alias("TotalAbsent")
).withColumn(
    "AttendanceRate",
    F.col("TotalPresent") / (F.col("TotalPresent") + F.col("TotalAbsent"))
)

df_rate.orderBy("AttendanceRate").show(25)

# Quick stats to see the actual spread
df_rate.select(
    F.min("AttendanceRate").alias("Min"),
    F.max("AttendanceRate").alias("Max"),
    F.avg("AttendanceRate").alias("Avg"),
    F.stddev("AttendanceRate").alias("StdDev")
).show()

+----------+------------+-----------+------------------+
|EmployeeID|TotalPresent|TotalAbsent|    AttendanceRate|
+----------+------------+-----------+------------------+
|      1016|         100|         15|0.8695652173913043|
|      1007|         106|         14|0.8833333333333333|
|      1012|         107|         14|0.8842975206611571|
|      1001|         110|         14|0.8870967741935484|
|      1020|         104|         13|0.8888888888888888|
|      1018|         109|         13|0.8934426229508197|
|      1011|         109|         13|0.8934426229508197|
|      1008|         110|         11|0.9090909090909091|
|      1002|         111|         11|0.9098360655737705|
|      1006|         111|         11|0.9098360655737705|
|      1015|         110|         10|0.9166666666666666|
|      1004|         107|          9|0.9224137931034483|
|      1003|         110|          8|0.9322033898305084|
|      1014|         112|          8|0.9333333333333333|
|      1010|         120|      